# Notebook 5: Tidsinhomogen koalescent og demografisk inferens

I denne notebook vil jeg undersøger, hvordan piecewise-konstante demografiske ændringer populationsflaskehalse, vækst og størrelsesskift former coalescent-fordelingen og SFS. Jeg bruger phasics trinvise (*distribution_context*) og epoch-baserede (*add_epoch*) tilgange til tidinhomogene modeller.

I en tidinhomogen koalescent varierer koalescensraten over tid som $\lambda(t) = \binom{k}{2}/N(t)$. Phasic approksimerer dette ved at lade mig ændre kantvægte på faste tidspunkter mens jeg integrerer CDF'en numerisk eller ved eksakt epochevis konstruktion.

In [ ]:
from phasic import Graph, with_ipv, StateIndexer, Property  # ALWAYS import phasic first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from functools import partial
from itertools import combinations_with_replacement
all_pairs = partial(combinations_with_replacement, r=2)
%config InlineBackend.figure_format = 'svg'

np.random.seed(42)
sns.set_palette('tab10')
plt.rcParams['figure.figsize'] = (10, 4)

# Hjælpefunktion: byg koalescent
nr_samples = 4

@with_ipv([nr_samples] + [0]*(nr_samples-1))
def coalescent_1param(state):
    transitions = []
    for i in range(state.size):
        for j in range(i, state.size):
            same = int(i == j)
            if same and state[i] < 2: continue
            if not same and (state[i] < 1 or state[j] < 1): continue
            new = state.copy()
            new[i] -= 1; new[j] -= 1; new[i+j+1] += 1
            transitions.append([new, [state[i]*(state[j]-same)/(1+same)]])
    return transitions

print("Setup færdig.")

## Del 1 – Populationsflaskehals: CDF-form og forventningsværdi

En flaskehals reducerer $N$ til $N_{\text{bottle}}$ i intervallet $[t_{\text{start}}, t_{\text{end}}]$, hvilket øger koalescensraten og fremskynder koalescens. Jeg implementerer dette via *distribution_context* og ændrer vægte på de rette tidspunkter.

### Hypotese 1

- En flaskehals forkorter E[TMRCA] og skaber en karakteristisk 'hump' i PDF'en ved $t_{\text{start}}$, fordi mange linjer koalescerer under flaskehalsen. Jo dybere flaskehalsen ($N_{\text{bottle}} \to 0$), desto mere punktmasse akkumuleres i intervallet $[t_{\text{start}}, t_{\text{end}}]$.

Jeg undersøger, hvordan CDF-formen ændrer sig systematisk med flaskehals-dybden $f = N/N_{\text{bottle}}$.

In [ ]:
def compute_bottleneck_cdf(N, N_bottle, t_start, t_end, cdf_cutoff=0.999, granularity=None):
    """Beregn CDF for en koalescent med populationsflaskehals."""
    graph = Graph(coalescent_1param)
    param_changes = [
        (t_start, [1/N_bottle]),
        (t_end,   [1/N])
    ]
    cdf_vals, times = [], []
    ctx = graph.distribution_context()
    graph.update_weights([1/N])
    for change_time, new_params in param_changes:
        while ctx.time() < change_time:
            cdf_vals.append(ctx.cdf())
            times.append(ctx.time())
            ctx.step()
            if ctx.cdf() >= cdf_cutoff: break
        graph.update_weights(new_params)
    while ctx.cdf() < cdf_cutoff:
        cdf_vals.append(ctx.cdf())
        times.append(ctx.time())
        ctx.step()
    # Expectation via accumulated occupancy
    acc_occ = graph.accumulated_occupancy(max(times)*2)
    E = float(np.sum(acc_occ))
    return np.array(times), np.array(cdf_vals), E

N = 1.0
t_start, t_end = 0.5, 0.9

# Baseline: ingen flaskehals
graph_base = Graph(coalescent_1param)
graph_base.update_weights([1/N])
t_vals = np.linspace(0, 6, 600)
cdf_base = graph_base.cdf(t_vals)

# Varierende flaskehals-dybde
depths = [1.0, 2.0, 5.0, 10.0, 50.0]  # f = N/N_bottle
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(t_vals, cdf_base, 'k--', lw=2, label='Ingen flaskehals')
axes[1].plot(t_vals, np.gradient(cdf_base, t_vals), 'k--', lw=2, label='Ingen flaskehals')

E_vals_depth = []
for f in depths:
    N_bottle = N / f
    t, cdf, E = compute_bottleneck_cdf(N, N_bottle, t_start, t_end)
    E_vals_depth.append({'Dybde f=N/N_b': f, 'N_bottle': round(N_bottle, 3), 'E[TMRCA]': round(E, 4)})
    axes[0].plot(t, cdf, label=f'f={f} (N_b={N_bottle:.2f})')
    pdf_approx = np.gradient(np.interp(t_vals, t, cdf), t_vals)
    axes[1].plot(t_vals, pdf_approx, label=f'f={f}')

for ax in axes:
    ax.axvspan(t_start, t_end, alpha=0.15, color='C1', label='Flaskehals-periode')

axes[0].set_xlabel('Tid'); axes[0].set_ylabel('CDF'); axes[0].set_title('CDF under flaskehals')
axes[0].legend(fontsize=8)
axes[1].set_xlabel('Tid'); axes[1].set_ylabel('PDF (approx.)'); axes[1].set_title('PDF under flaskehals')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(pd.DataFrame(E_vals_depth).to_string(index=False))

### Hypotese 2 – Flaskehals-timing afgør, om den er synlig i SFS

- En sen flaskehals ($t_{\text{start}}$ nær TMRCA) påvirker primært singleton gren længde, mens en tidlig flaskehals primært påvirker højere-ordens grene. SFS-formen er sensitiv over for flaskehalstidspunktet men ikke over for flaskehalsdybden, når dybden er stor.

In [ ]:
def bottleneck_accumulated_sfs(N, N_bottle, t_start, t_end, t_max=8):
    """Beregn SFS via accumulated_occupancy under en flaskehals."""
    graph = Graph(coalescent_1param)
    param_changes = [(t_start, [1/N_bottle]), (t_end, [1/N])]
    ctx = graph.distribution_context()
    graph.update_weights([1/N])
    for change_time, new_params in param_changes:
        while ctx.time() < change_time:
            ctx.step()
            if ctx.cdf() > 0.9999: break
        graph.update_weights(new_params)
    while ctx.cdf() < 0.9999:
        ctx.step()
    acc_occ = graph.accumulated_occupancy(t_max)
    reward_matrix = graph.states().T
    sfs = [float(np.sum(acc_occ * reward_matrix[i])) for i in range(nr_samples-1)]
    return sfs

N, N_bottle = 1.0, 0.1  # Fast dybde
labels = [f"{i+1}'ton" for i in range(nr_samples-1)]
timings = [0.3, 0.6, 1.0, 1.5, 2.5]  # t_start varierer

# Baseline SFS
graph_base.update_weights([1/N])
sfs_base = [graph_base.expectation(graph_base.states().T[i]) for i in range(nr_samples-1)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Subplot 1: SFS som funktion af t_start
x = np.arange(len(labels))
width = 0.12
axes[0].bar(x, sfs_base, width, label='Ingen flaskehals', color='black', alpha=0.6)
for idx, t_s in enumerate(timings):
    sfs = bottleneck_accumulated_sfs(N, N_bottle, t_s, t_s+0.4)
    axes[0].bar(x + (idx+1)*width, sfs, width, label=f't_start={t_s}')
axes[0].set_xticks(x + width*3)
axes[0].set_xticklabels(labels)
axes[0].set_ylabel('E[branch length]'); axes[0].set_title('SFS: effekt af flaskehalstidspunkt')
axes[0].legend(fontsize=7)

# Subplot 2: Relativ ændring i SFS som funktion af t_start
for i, lab in enumerate(labels):
    rel_changes = []
    for t_s in timings:
        sfs = bottleneck_accumulated_sfs(N, N_bottle, t_s, t_s+0.4)
        rel_changes.append((sfs[i] - sfs_base[i]) / sfs_base[i])
    axes[1].plot(timings, rel_changes, 'o-', label=lab)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_xlabel('t_start (flaskehalstidspunkt)'); axes[1].set_ylabel('Relativ ændring i E[branch length]')
axes[1].set_title('Relativ SFS-ændring vs. flaskehalstidspunkt')
axes[1].legend()
plt.tight_layout(); plt.show()

## Del 2 – Multi-epoch demografisk model

Jeg udvider til en serie af epoker med forskellig $N_e$, der svarer til en typisk demografisk scenarie: vækst, stabilitet, flaskehals, ekspansion.

### Hypotese 3

- En multi-epoch model med voksende $N_e$ over tid (eksponentiel vækst baglæns) giver en bimodal eller højreskæv TMRCA-fordeling, fordi tidlige linjer har høj koalescensrate (lille $N_e$ i nutid), mens sen koalescens er sjælden (stor $N_e$ i fortiden). Dette er karakteristisk for ekspanderende populationer.

In [ ]:
def multi_epoch_cdf(epoch_boundaries, pop_sizes, cdf_cutoff=0.9999):
    """CDF for multi-epoch model med piecewise konstant N_e."""
    assert len(epoch_boundaries) == len(pop_sizes) - 1, "En grænse per overgang"
    graph = Graph(coalescent_1param)
    graph.update_weights([1/pop_sizes[0]])
    changes = [(t, [1/N]) for t, N in zip(epoch_boundaries, pop_sizes[1:])]
    cdf_vals, times = [], []
    ctx = graph.distribution_context()
    for change_time, new_params in changes:
        while ctx.time() < change_time:
            cdf_vals.append(ctx.cdf())
            times.append(ctx.time())
            ctx.step()
            if ctx.cdf() >= cdf_cutoff: break
        graph.update_weights(new_params)
    while ctx.cdf() < cdf_cutoff:
        cdf_vals.append(ctx.cdf())
        times.append(ctx.time())
        ctx.step()
    return np.array(times), np.array(cdf_vals)

# Fire demografiske scenarier
scenarios = {
    'Konstant (N=1)':        {'bounds': [1, 2, 4], 'sizes': [1, 1, 1, 1]},
    'Ekspansion (N vokser)': {'bounds': [1, 2, 4], 'sizes': [0.2, 0.5, 1.0, 5.0]},
    'Kontraktion (N falder)':{'bounds': [1, 2, 4], 'sizes': [5.0, 1.0, 0.5, 0.2]},
    'Flaskehals + Recovery': {'bounds': [0.5, 1.5, 3], 'sizes': [1, 0.1, 0.1, 1]},
    'Stepwise vækst':        {'bounds': [0.5, 1.0, 2.0], 'sizes': [0.25, 0.5, 1.0, 2.0]},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
t_fine = np.linspace(0, 8, 800)

for name, sc in scenarios.items():
    t, cdf = multi_epoch_cdf(sc['bounds'], sc['sizes'])
    cdf_interp = np.interp(t_fine, t, cdf)
    pdf_approx = np.gradient(cdf_interp, t_fine)
    axes[0].plot(t_fine, cdf_interp, label=name)
    axes[1].plot(t_fine, np.maximum(pdf_approx, 0), label=name)

# Markér epoker for flaskehals-scenariet
for b in [0.5, 1.5, 3]:
    axes[0].axvline(b, color='gray', linestyle=':', alpha=0.5)
    axes[1].axvline(b, color='gray', linestyle=':', alpha=0.5)

axes[0].set_xlabel('Tid'); axes[0].set_ylabel('CDF'); axes[0].set_title('TMRCA-CDF under demografiske scenarier')
axes[0].legend(fontsize=7)
axes[1].set_xlabel('Tid'); axes[1].set_ylabel('PDF (approx.)'); axes[1].set_title('TMRCA-PDF under demografiske scenarier')
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## Del 3 – SFS-dynamik over tid: Heatmap

Med *accumulated_occupancy(t)* kan jeg beregne, hvor meget SFS'en er akkumuleret op til tid $t$. Dette giver mig en dynamisk visning af, hvordan SFS'en bygges op og hvordan den ændrer sig under demografiske hændelser.

### Hypotese 4

- Under en flaskehals accelereres akkumuleringen af **alle** SFS-komponenter, men effekten er størst for singletons (1-tons), fordi de fleste linjer er singletons tidligt i koalescensen. Flaskehalsen manifesterer sig som et pludseligt horisontalt bånd i heatmappet.

In [ ]:
def sfs_heatmap_over_time(N, N_bottle, t_start, t_end, t_max=5, n_steps=200):
    """Beregn normaliseret SFS akkumuleret til hvert tidspunkt t."""
    graph = Graph(coalescent_1param)
    reward_matrix = graph.states().T
    times = np.linspace(0.01, t_max, n_steps)
    result = np.zeros((nr_samples-1, n_steps))
    # Setup
    param_changes = [(t_start, [1/N_bottle]), (t_end, [1/N])]
    ctx = graph.distribution_context()
    graph.update_weights([1/N])
    for change_time, new_params in param_changes:
        while ctx.time() < change_time:
            ctx.step()
        graph.update_weights(new_params)
    for j, t in enumerate(times):
        acc_occ = graph.accumulated_occupancy(t)
        col = np.array([float(np.sum(acc_occ * reward_matrix[i])) for i in range(nr_samples-1)])
        col_sum = col.sum()
        result[:, j] = col / col_sum if col_sum > 0 else col
    return times, result

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, N_b) in zip(axes, [('Ingen flaskehals (N_b=N)', 1.0), ('Dyb flaskehals (N_b=N/20)', 0.05)]):
    times, result = sfs_heatmap_over_time(1.0, N_b, 0.5, 0.9, t_max=5, n_steps=150)
    ton_labels = [f"{i+1}'ton" for i in range(nr_samples-1)]
    sns.heatmap(
        pd.DataFrame(result, index=ton_labels, columns=times.round(2)),
        ax=ax, cmap='viridis',
        xticklabels=30, cbar_kws={'label': 'Normaliseret andel'}
    )
    ax.axvline(x=np.searchsorted(times, 0.5), color='red', lw=2, linestyle='--', label='Flaskehals start')
    ax.axvline(x=np.searchsorted(times, 0.9), color='orange', lw=2, linestyle='--', label='Flaskehals slut')
    ax.set_title(title); ax.set_xlabel('Tid'); ax.set_ylabel('SFS-komponent')
    ax.invert_yaxis()

plt.suptitle('Normaliseret SFS akkumuleret over tid: med og uden flaskehals', fontsize=12)
plt.tight_layout(); plt.show()

## Del 4 – Epoch-baserede momenter og sammenligning med analytisk resultat

Jeg bruger epoch-modellen (*add_epoch*) til at beregne eksakte momenter for en 2-epoch demografisk model og sammenligner med den analytiske approksimation.

### Hypotese 5

- Den eksakte epoch-baserede beregning af E[TMRCA] stemmer overens med den numeriske approksimation fra *distribution_context* til mindst 4 decimaler, men epoch-tilgangen er betydeligt hurtigere for modeller med mange epoker, fordi hvert trin i *distribution_context* integrerer numerisk.

In [ ]:
import time

# Epoch-baseret model (eksakt, fra vejleder)
nr_samples_ep = 2
epochs_list = [0.0, 1.0, 3.0]  # grænser
pop_sizes_ep = [1.0, 5.0, 1.0]  # N_e i hvert interval

indexer_ep = StateIndexer(
    lineages=[Property('ton', min_value=1, max_value=nr_samples_ep)],
    slots=['epoch']
)

def coalescent_epoch(state, epochs=None, epoch_idx=None, indexer=None):
    transitions = []
    epoch_idx = int(epoch_idx)
    if state[indexer.epoch] != epoch_idx:
        return transitions
    for i, j in all_pairs(indexer.lineages):
        pi = indexer.lineages.index_to_props(i)
        pj = indexer.lineages.index_to_props(j)
        if state.sum() <= 1: continue
        same = int(pi.ton == pj.ton)
        if same and state[i] < 2: continue
        if not same and (state[i] < 1 or state[j] < 1): continue
        new = state.copy()
        new[i] -= 1; new[j] -= 1
        k = indexer_ep.props_to_index(ton=pi.ton + pj.ton)
        new[k] += 1
        coeff = np.zeros(len(pop_sizes_ep) + 1)
        coeff[epoch_idx] = state[i] * (state[j] - same) / (1 + same)
        transitions.append([new, coeff])
    return transitions

# Byg epoch-modellen
ipv_ep = [0] * indexer_ep.state_length
ipv_ep[indexer_ep.props_to_index(ton=1)] = nr_samples_ep

def add_epoch(graph, epoch_idx, indexer_ep, epochs_list):
    """Tilføj en ny epoch til grafen."""
    t_boundary = epochs_list[epoch_idx]
    state_transfer_rate = 1.0  # instant transfer
    # Find tilstande fra forrige epoch og tilslut til ny
    for v_idx in range(1, graph.vertices_length()):
        v = graph.vertex_at(v_idx)
        s = v.state()
        if s[indexer_ep.epoch] == epoch_idx - 1:
            new_s = s.copy()
            new_s[indexer_ep.epoch] = epoch_idx
            target = graph.find_or_create_vertex(new_s)
            coeff = np.zeros(len(pop_sizes_ep) + 1)
            coeff[-1] = state_transfer_rate
            v.add_edge(target, coeff)

t0 = time.perf_counter()
graph_ep = Graph(coalescent_epoch, ipv=ipv_ep, epoch_idx=0,
                 epochs=epochs_list, indexer=indexer_ep)
for ep_idx in range(1, len(epochs_list)):
    add_epoch(graph_ep, ep_idx, indexer_ep, epochs_list)
weights = [1/N for N in pop_sizes_ep] + [1.0 / (epochs_list[1] - epochs_list[0]) if len(epochs_list) > 1 else 1.0]
graph_ep.update_weights(weights)
t_epoch = time.perf_counter() - t0
E_epoch = graph_ep.expectation()

# Numerisk approksimation via distribution_context
t0 = time.perf_counter()
@with_ipv([nr_samples_ep] + [0]*(nr_samples_ep-1))
def coal_2samp(state):
    transitions = []
    for i in range(state.size):
        for j in range(i, state.size):
            same = int(i == j)
            if same and state[i] < 2: continue
            if not same and (state[i] < 1 or state[j] < 1): continue
            new = state.copy()
            new[i] -= 1; new[j] -= 1; new[i+j+1] += 1
            transitions.append([new, [state[i]*(state[j]-same)/(1+same)]])
    return transitions

g_ctx = Graph(coal_2samp)
changes = [(epochs_list[i], [1/pop_sizes_ep[i]]) for i in range(1, len(epochs_list))]
g_ctx.update_weights([1/pop_sizes_ep[0]])
ctx = g_ctx.distribution_context()
for ct, p in changes:
    while ctx.time() < ct:
        ctx.step()
    g_ctx.update_weights(p)
while ctx.cdf() < 0.9999:
    ctx.step()
E_ctx = float(np.sum(g_ctx.accumulated_occupancy(ctx.time()*1.5)))
t_ctx = time.perf_counter() - t0

print(f"Epoch-model E[TMRCA]  = {E_epoch:.6f}  (tid: {t_epoch*1000:.1f} ms)")
print(f"Numerisk approx.     = {E_ctx:.6f}  (tid: {t_ctx*1000:.1f} ms)")
print(f"Absolut afvigelse    = {abs(E_epoch - E_ctx):.2e}")
print(f"\nEpoch-model er {t_ctx/t_epoch:.1f}x hurtigere end numerisk approksimation")

## Del 5 – Laplace-transformation: Prob. for ingen rekombination

Laplace-transformationen $\mathcal{L}\{f\}(s) = E[e^{-s\tau}]$ giver mig sandsynligheden for, at ingen begivenhed med rate $s$ er sket inden absorption. For en rekombinationsrate $\rho$ giver dette sandsynligheden for, at to loci deler den samme genealogi (ingen rekombination på grenene).

### Hypotese 6

- Sandsynligheden for fælles genealogi $P(\text{ingen rek.})$ falder hurtigt med rekombinationsafstanden $\rho$, men falder langsommere for store populationer ($N$ stor), fordi koalescensen sker sent og rekombinationen har lang tid at operere i. For en bottleneck-population er $P$ højere end for en konstant-N population ved samme $\rho$

In [ ]:
# Standard koalescent og Laplace transformation
graph_laplace = Graph(coalescent_1param)
rho_values = np.logspace(-2, 1, 60)  # rekombinationsrate

# Tre populationsstørrelser
N_scenarios = [0.5, 1.0, 3.0]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for N in N_scenarios:
    graph_laplace.update_weights([1/N])
    probs = []
    for rho in rho_values:
        lap_graph = graph_laplace.laplace_transform(rho)
        rewards = graph_laplace.absorbing_state_rewards()
        p = lap_graph.expectation(rewards)
        probs.append(float(p))
    axes[0].semilogx(rho_values, probs, label=f'N={N}')
    axes[1].loglog(rho_values, probs, label=f'N={N}')

axes[0].set_xlabel('Rekombinationsrate ρ (log)'); axes[0].set_ylabel('P(ingen rekombination)')
axes[0].set_title('Sandsynlighed for fælles genealogi')
axes[0].legend()
axes[1].set_xlabel('Rekombinationsrate ρ (log)'); axes[1].set_ylabel('P(ingen rek.) (log)')
axes[1].set_title('Log-log plot: eksponentiel vs. power-law')
axes[1].legend()
plt.suptitle('Laplace-transformation: P(ingen rekombination) som funktion af ρ', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Effekt af flaskehals på P(ingen rekombination)
rho_test = np.logspace(-2, 1, 40)
fig, ax = plt.subplots(figsize=(8, 5))

# Baseline N=1
graph_laplace.update_weights([1.0])
p_base = [float(graph_laplace.laplace_transform(r).expectation(graph_laplace.absorbing_state_rewards())) for r in rho_test]
ax.semilogx(rho_test, p_base, 'k--', lw=2, label='Konstant N=1')

# Flaskehals-scenarier: vi approksimerer med effektiv N
# Flaskehalsen reducerer E[TMRCA], og vi sammenligner P(ingen rek.)
for f in [2, 5, 10]:
    N_bottle = 1.0/f
    # Effektiv rate = vægtet gennemsnit (simpel approx)
    # For eksakt: brug distribution_context
    graph_laplace.update_weights([f])  # Hurtigere koalescens
    p_vals = [float(graph_laplace.laplace_transform(r).expectation(graph_laplace.absorbing_state_rewards())) for r in rho_test]
    ax.semilogx(rho_test, p_vals, label=f'Flaskehals (f={f})')

ax.set_xlabel('Rekombinationsrate ρ'); ax.set_ylabel('P(ingen rekombination)')
ax.set_title('Flaskehalsdybde og sandsynlighed for fælles genealogi')
ax.legend()
plt.tight_layout(); plt.show()